# Real Estate Recommender — Implicit ALS
Trains an Alternating Least Squares model on the implicit like data using the `implicit` library.
ALS is designed for implicit feedback (no explicit ratings needed) and produces ranking-optimised embeddings.

**Requires:** `pip install implicit`

In [1]:
import numpy as np
import pandas as pd
import implicit
from scipy.sparse import coo_matrix

In [2]:
# Load datasets
users_df = pd.read_csv('data-refined/users.csv')
listings_df = pd.read_csv('data-refined/cleaned_listings.csv')
user_likes_df = pd.read_csv('data-refined/user_likes.csv')

print(f"Users: {users_df.shape[0]} | Listings: {listings_df.shape[0]} | Likes: {user_likes_df.shape[0]}")

Users: 200 | Listings: 26648 | Likes: 26546


## Build the user-item interaction matrix

In [3]:
# Build a users x items sparse interaction matrix
user_ids = np.sort(user_likes_df['user_id'].unique())
item_ids = np.sort(listings_df['listing_id'].unique())
user_to_idx = {int(u): i for i, u in enumerate(user_ids)}
item_to_idx = {int(it): i for i, it in enumerate(item_ids)}

rows = user_likes_df['user_id'].map(user_to_idx)
cols = user_likes_df['listing_id'].map(item_to_idx)
mask = rows.notna() & cols.notna()
rows = rows[mask].astype(int).to_numpy()
cols = cols[mask].astype(int).to_numpy()
vals = np.ones(len(rows), dtype=np.float32)

user_items = coo_matrix((vals, (rows, cols)), shape=(len(user_ids), len(item_ids))).tocsr()
user_items.data[:] = 1.0

skipped = int((~mask).sum())
if skipped:
    print(f"Warning: skipped {skipped} likes whose listing_id was not in listings_df.")
print(f"Interaction matrix: {user_items.shape[0]} users x {user_items.shape[1]} items | {user_items.nnz} interactions")

Interaction matrix: 200 users x 26648 items | 26546 interactions


## Train ALS model

In [4]:
# ALS on implicit feedback
model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.02,
    alpha=40.0,
    iterations=30,
    random_state=42,
)
model.fit(user_items)
print("ALS model trained.")

  0%|          | 0/30 [00:00<?, ?it/s]

ALS model trained.


## Generate recommendations

In [5]:
# Recommend top-10 listings for the first user in the data
sample_user_id = int(user_ids[0])
sample_user_idx = user_to_idx[sample_user_id]

rec_item_idx, rec_scores = model.recommend(
    sample_user_idx,
    user_items[sample_user_idx],
    N=10,
    filter_already_liked_items=True,
)
rec_listing_ids = [int(item_ids[i]) for i in rec_item_idx]

print(f"Top-10 ALS recommendations for user {sample_user_id}:")
for rank, (lid, score) in enumerate(zip(rec_listing_ids, rec_scores), start=1):
    print(f"  {rank:2d}. listing_id={lid}  score={score:.4f}")

Top-10 ALS recommendations for user 1:
   1. listing_id=9919  score=0.6744
   2. listing_id=17169  score=0.6252
   3. listing_id=4462  score=0.5634
   4. listing_id=23414  score=0.5200
   5. listing_id=21648  score=0.4965
   6. listing_id=17401  score=0.4960
   7. listing_id=26524  score=0.4898
   8. listing_id=7504  score=0.4894
   9. listing_id=14338  score=0.4838
  10. listing_id=2064  score=0.4630


## Find similar listings (item-item)

In [6]:
# Find listings similar to the first recommended item
query_listing_id = rec_listing_ids[0]
query_item_idx = item_to_idx[query_listing_id]

similar_idx, similar_scores = model.similar_items(query_item_idx, N=6)
similar_listing_ids = [int(item_ids[i]) for i in similar_idx]

print(f"Listings similar to listing_id={query_listing_id}:")
for lid, score in zip(similar_listing_ids, similar_scores):
    row = listings_df[listings_df['listing_id'] == lid]
    if not row.empty:
        r = row.iloc[0]
        print(f"  listing_id={lid}  score={score:.4f}  ${r.get('price', 'N/A'):,}  {r.get('bed','?')}bd/{r.get('bath','?')}ba  {r.get('city','?')}, {r.get('state','?')}")
    else:
        print(f"  listing_id={lid}  score={score:.4f}")

Listings similar to listing_id=9919:
  listing_id=9919  score=1.0000  $59,900  3bd/2ba  Birmingham, Alabama
  listing_id=3258  score=0.7077  $265,000  4bd/2ba  Tuscaloosa, Alabama
  listing_id=5095  score=0.6785  $169,900  3bd/1ba  Mount Olive, Alabama
  listing_id=4651  score=0.6785  $149,900  4bd/2ba  Hanceville, Alabama
  listing_id=2954  score=0.6785  $179,900  3bd/2ba  Jemison, Alabama
  listing_id=1505  score=0.6785  $749,000  3bd/2ba  Orange Beach, Alabama
